# Historical S&P 500 dataset for Phase 4

This notebook builds a stock-month dataset from reconstructed historical S&P 500 membership and adjusted prices from 2018–2024. Features and membership use the trading day before the formation close; the label is the subsequent holding-period return. Data gaps, ticker mapping, corporate actions, and membership reconstruction need further independent checks.


In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
root = Path.cwd()
if not (root / "src").is_dir(): root = root.parent
sys.path.insert(0, str(root))
from src.data_loader import load_historical_prices, load_historical_membership, constituents_as_of
data = root / "data/sp500_historical_2018_2024"
prices = load_historical_prices(data / "prices_long.parquet")
membership = load_historical_membership(data / "members.parquet")
metadata = json.loads((data / "metadata.json").read_text())
print("Source:", metadata["source_repository"])
print("Price range:", prices.index.min().date(), "to", prices.index.max().date())
print("Historical symbols:", len(prices.columns), "| Membership dates:", membership.date.nunique())
membership.head()


## Inspect missing history

A stock can lack prices because it was not yet listed, its ticker changed, or a download failed. These cases must not be filled as if the price stayed constant. The coverage table is diagnostic, not proof that a missing record is a genuine absence.

In [ ]:
coverage = pd.DataFrame({"ticker": prices.columns, "observations": prices.notna().sum().values})
print("Symbols with at least one price:", (coverage.observations > 0).sum())
print("Missing daily price cells:", f"{prices.isna().mean().mean():.1%}")
coverage.sort_values("observations").head(10)


## Build the learning table

Feature values and membership are known by the trading day before formation. Labels measure returns from the formation close to the next month-end close. Observations without complete inputs or labels are removed; investigate missingness and delisting outcomes before using these rows for conclusions.


In [ ]:
daily_returns = prices.pct_change(fill_method=None)
month_prices = prices.resample("ME").last(skipna=False)
valid_momentum = prices.notna().rolling(253, min_periods=253).sum().eq(253)
momentum = prices.pct_change(252, fill_method=None).where(valid_momentum).shift(1).resample("ME").last(skipna=False)
volatility = (daily_returns.rolling(252, min_periods=252).std() * np.sqrt(252)).shift(1).resample("ME").last(skipna=False)
next_return = month_prices.shift(-1).div(month_prices).sub(1)
def to_long(frame, name):
    return frame.rename_axis(index="formation_date", columns="ticker").stack().rename(name)
features = pd.concat([to_long(momentum, "momentum_252d"), to_long(volatility, "volatility_252d"), to_long(next_return, "next_month_return")], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
feature_dates = features.index.get_level_values("formation_date")
feature_tickers = features.index.get_level_values("ticker")
monthly_members = {d: set(constituents_as_of(membership, prices.loc[:d].index[-2])) for d in feature_dates.unique()}
features = features.loc[[ticker in monthly_members[date] for date, ticker in zip(feature_dates, feature_tickers)]]
print(f"Complete stock-month observations: {len(features):,}")
print("Stocks:", features.index.get_level_values("ticker").nunique())
print("Date range:", features.index.get_level_values("formation_date").min().date(), "to", features.index.get_level_values("formation_date").max().date())
features.head()


## Chronological split

Use formation dates through November 2021 for development and January 2022 onward for model holdout. December 2021 is excluded because its label ends in January 2022. Split all stocks by date together. Phase 3 results for these holdout years were already inspected; a later evaluation period would be more independent.


In [ ]:
dates = features.index.get_level_values("formation_date")
train = features.loc[dates < "2021-12-01"].copy()
holdout = features.loc[dates >= "2022-01-01"].copy()
assert train.index.get_level_values("formation_date").max() <= pd.Timestamp("2021-11-30")
assert holdout.index.get_level_values("formation_date").min() >= pd.Timestamp("2022-01-01")
print(f"Development: {len(train):,} rows | holdout: {len(holdout):,} rows")
print(f"Development dates: {train.index.get_level_values('formation_date').min().date()} to {train.index.get_level_values('formation_date').max().date()}")
print(f"Holdout dates: {holdout.index.get_level_values('formation_date').min().date()} to {holdout.index.get_level_values('formation_date').max().date()}")